# **TechMart Kaggle Factory — Building the Hybrid AI Artifacts**

### **Project Type**    — Offline Model Factory / RAG Index Build / Text-Classification Fine-Tuning

This notebook is the **"Kaggle Factory"** for the *TechMart Multi-Agent AI Customer
Support Assistant*. It performs the heavy, GPU-bound work **once, offline**, and
emits two production artifacts that the lightweight FastAPI backend loads at
startup for fast, cheap, local inference.

> Companion docs in the repo: `documentations/13_KAGGLE_FACTORY_PATTERN.md`
> (pattern) and `documentations/14_KAGGLE_FACTORY_NOTEBOOK_BUILD.md` (this build).

# **Project Summary**

Modern AI applications have two very different workloads. **Heavy-lifting**
(training a classifier, embedding an entire knowledge base) is batchy, slow, and
GPU-hungry. **Lightweight inference** (embedding one query, classifying one
message) must be fast and cheap. Running the heavy work on a live server is slow,
costly, and — using a general LLM for every intent decision — hits rate limits
(HTTP 429).

The **Kaggle Factory** pattern decouples them: this notebook (GPU) builds the
artifacts; production (CPU) just loads and serves them.

```
   ┌─────────────────────────────┐        ┌──────────────────────────────┐
   │   KAGGLE FACTORY (offline)  │        │   PRODUCTION APP (online)    │
   │   GPU · Internet · batch    │  ===>  │   CPU · stateless · fast     │
   │                             │ build  │                              │
   │  A1 · RAG index artifact    │ ─────► │  loads at startup:           │
   │  A2 · Intent router artifact│        │   • vectorstore/faiss_index/ │
   └─────────────────────────────┘        │   • backend/models/          │
                                          │       intent_classifier/     │
                                          │  LLM used ONLY for the final │
                                          │  response generation         │
                                          └──────────────────────────────┘
```

# **What We Build (the two artifacts)**

| # | Artifact | Built from | Output | Loaded in production by |
|---|----------|-----------|--------|-------------------------|
| **A1** | **RAG index** | `knowledge_base/` PDFs | `faiss_index/index.faiss` + `metadata.pkl` | `backend/rag/retriever.py` |
| **A2** | **Intent router** | Banking77 + CFPB + SQuAD/MS MARCO | `intent_classifier/` (HF model dir) | `backend/agents/intent_classifier.py` |

### ⚠️ Format contracts (must match production exactly)
- **A1:** embed with `sentence-transformers/all-MiniLM-L6-v2`; index type
  `IndexFlatL2`; chunk size **500** / overlap **50**; each metadata row is a dict
  with keys `source, page, chunk_id, text` aligned to the i-th vector.
- **A2:** Hugging Face `text-classification` format; **5** labels with
  `id2label = {0:billing, 1:technical, 2:product, 3:complaint, 4:faq}`.

If any of these drift from the production loaders, retrieval/routing breaks
silently — so they are enforced throughout this notebook.

# **Prerequisites & Session Settings**

1. **Accelerator:** `GPU T4 x2` (or `P100`) — Settings panel → *Accelerator*.
2. **Internet:** **On** — needed for `pip install` and to pull the base models.
3. **Attach datasets** (Add Data):
   - `techmart-knowledge-base` — the 8 PDFs → appears at `/kaggle/input/techmart-knowledge-base/`
   - `techmart-intent-data` — `banking77/*.csv`, `cfpb_sample.csv`, `squad/*.json`,
     `msmarco/queries.dev.small.tsv` → `/kaggle/input/techmart-intent-data/`
4. *(Optional)* Add-ons → **Secrets** → `OPENROUTER_API_KEY` (only to smoke-test
   final LLM generation; not needed to build the artifacts).

Every load below uses **adaptive `/kaggle/input/` discovery** (glob), so the exact
dataset slugs do not need to be hard-coded and missing optional files are skipped
gracefully.

# **Let's Begin!**

## 1. Imports & Configuration

Install the RAG + fine-tuning stack, import everything up front, silence noisy
warnings, and create the two output directories under `/kaggle/working/` (which
Kaggle captures on *Save Version*).

In [ ]:
# Install the pipeline stack (Kaggle base image already ships torch/sklearn)
!pip install -q langchain langchain-community langchain-text-splitters sentence-transformers faiss-gpu transformers datasets pypdf accelerate

In [ ]:
# ============================================================
# SECTION 1: IMPORTS & CONFIGURATION
# All libraries required for the full Factory pipeline
# ============================================================
import os
import re
import glob
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import torch

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# ── Config constants (MUST match the production loaders) ────────────────────
EMBED_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
BASE_LM       = 'distilbert-base-uncased'
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
DOMAINS       = ['billing', 'technical', 'product', 'complaint', 'faq']
ID2LABEL      = {i: d for i, d in enumerate(DOMAINS)}
LABEL2ID      = {d: i for i, d in enumerate(DOMAINS)}

# ── Output directories (the artifacts) ──────────────────────────────────────
FAISS_OUT = '/kaggle/working/faiss_index'
CLF_OUT   = '/kaggle/working/intent_classifier'
os.makedirs(FAISS_OUT, exist_ok=True)
os.makedirs(CLF_OUT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('✅ Environment ready')
print(f'   torch      : {torch.__version__}')
print(f'   CUDA avail : {torch.cuda.is_available()}  ->  device = {DEVICE}')
print(f'   Domains    : {DOMAINS}')
print(f'   Artifacts  : {FAISS_OUT}  |  {CLF_OUT}')

## 2. Load the Knowledge Base (PDFs)

**Why this step?** The RAG index can only answer from documents it has seen. We
discover every PDF under `/kaggle/input/` adaptively (so the dataset slug is not
hard-coded), read each page's text, and skip empty pages.

In [ ]:
# ============================================================
# SECTION 2: LOAD KNOWLEDGE-BASE PDFs
# Adaptive discovery — no hard-coded dataset slug
# ============================================================
pdf_paths = sorted(glob.glob('/kaggle/input/**/*.pdf', recursive=True))
# Fallback for local runs
if not pdf_paths:
    pdf_paths = sorted(glob.glob('knowledge_base/*.pdf'))

assert pdf_paths, 'No PDFs found — attach the techmart-knowledge-base dataset.'
print(f'📚 Found {len(pdf_paths)} PDF(s):')
for p in pdf_paths:
    print('   -', os.path.basename(p))

pages = []  # (source_filename, page_number, text)
for path in pdf_paths:
    reader = PdfReader(path)
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ''
        if text.strip():
            pages.append((os.path.basename(path), page_num + 1, text))

print(f'\n✅ Extracted {len(pages)} non-empty pages')

## 3. Chunk the Documents

**Why this step?** LLMs and embeddings work best on focused passages. We split
each page with `RecursiveCharacterTextSplitter` using **chunk_size=500 /
overlap=50** — identical to `backend/rag/pipeline.py`, so Factory chunks match
what production expects. Each chunk becomes a metadata dict with the exact keys
the retriever reads: `source, page, chunk_id, text`.

In [ ]:
# ============================================================
# SECTION 3: CHUNK DOCUMENTS  (contract: source/page/chunk_id/text)
# ============================================================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

all_chunks, all_metadata = [], []
for source, page_num, text in pages:
    stem = os.path.splitext(source)[0]
    for i, chunk in enumerate(splitter.split_text(text)):
        all_chunks.append(chunk)
        all_metadata.append({
            'source':   source,
            'page':     page_num,
            'chunk_id': f'{stem}_{page_num}_{i}',
            'text':     chunk,
        })

print(f'✅ Built {len(all_chunks)} chunks from {len(pages)} pages')
print('   sample metadata:', {k: (v[:50] + "…" if k == "text" else v)
                              for k, v in all_metadata[0].items()})

## 4. Build Embeddings (GPU)

**Why this step?** This is the heavy, GPU-justifying part of Artifact 1 —
converting every chunk into a 384-dim vector with the **same** embedding model
production uses for queries. Mismatched models = meaningless similarity.

In [ ]:
# ============================================================
# SECTION 4: EMBED CHUNKS ON GPU  (all-MiniLM-L6-v2, dim=384)
# ============================================================
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
embeddings = embedder.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,      # progress bar is fine offline
    convert_to_numpy=True,
    normalize_embeddings=False,  # production uses raw L2 distance
).astype('float32')

print(f'✅ Embeddings shape: {embeddings.shape}  (dim={embeddings.shape[1]})')

## 5. Build & Save the FAISS Index  ──  **ARTIFACT 1**

**Why this step?** `IndexFlatL2` (exact L2 search) matches
`backend/rag/retriever.py`. We persist `index.faiss` and the aligned
`metadata.pkl` — the complete RAG artifact.

In [ ]:
# ============================================================
# SECTION 5: BUILD + SAVE FAISS INDEX  ── ARTIFACT 1
# ============================================================
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
assert index.ntotal == len(all_metadata), 'vector/metadata count mismatch!'

faiss.write_index(index, os.path.join(FAISS_OUT, 'index.faiss'))
with open(os.path.join(FAISS_OUT, 'metadata.pkl'), 'wb') as f:
    pickle.dump(all_metadata, f)

for fn in ['index.faiss', 'metadata.pkl']:
    fp = os.path.join(FAISS_OUT, fn)
    print(f'✅ Saved {fp:<45} ({os.path.getsize(fp)/1024:.1f} KB)')
print(f'\n✅ ARTIFACT 1 complete — {index.ntotal} vectors indexed')

## 6. Load & Explore the Intent Data

**Why this step?** Artifact 2 is a supervised classifier, so it needs labeled
text. We pull from up to four sources, each discovered adaptively and skipped
gracefully if absent:

| Source | Role | Gives us |
|--------|------|----------|
| **Banking77** (`train/test.csv`) | 77 labeled intents | billing / technical / product / faq signal |
| **CFPB** (`cfpb_sample.csv` or `complaints.csv`) | real complaint narratives | the **complaint** class |
| **SQuAD** (`*.json`) + **MS MARCO** (`*.tsv`) | general questions | **faq** augmentation |

In [ ]:
# ============================================================
# SECTION 6: LOAD RAW INTENT DATA  (adaptive + graceful)
# ============================================================
def _find(pattern):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        hits = glob.glob(f'datasets/**/{pattern}', recursive=True)
    return hits

# ── Banking77 (labeled) ─────────────────────────────────────
b77_files = _find('train.csv') + _find('test.csv')
b77_files = [f for f in b77_files if 'banking' in f.lower()] or _find('*.csv')
banking77 = pd.concat([pd.read_csv(f) for f in set(b77_files)
                       if {'text', 'category'}.issubset(pd.read_csv(f, nrows=1).columns)],
                      ignore_index=True) if b77_files else pd.DataFrame(columns=['text', 'category'])
print(f'Banking77 rows : {len(banking77):,}  | intents: {banking77["category"].nunique() if len(banking77) else 0}')

# ── CFPB complaints (sample safely; never load the full 8.9 GB) ──
cfpb_hits = _find('cfpb_sample.csv') or _find('complaints.csv')
cfpb_texts = []
if cfpb_hits:
    cfpb = pd.read_csv(cfpb_hits[0],
                       usecols=lambda c: c in ['Product', 'Issue', 'Consumer complaint narrative'],
                       nrows=200_000)
    narr = cfpb.get('Consumer complaint narrative')
    if narr is not None:
        cfpb_texts = narr.dropna().astype(str).str.strip()
        cfpb_texts = cfpb_texts[cfpb_texts.str.len() > 20].tolist()
print(f'CFPB narratives: {len(cfpb_texts):,}')

# ── SQuAD + MS MARCO questions (unlabeled → faq augmentation) ──
faq_questions = []
for sq in _find('*.json'):
    if 'squad' in sq.lower() or 'dev-v' in sq.lower() or 'train-v' in sq.lower():
        try:
            data = json.load(open(sq))['data']
            for art in data:
                for para in art['paragraphs']:
                    for qa in para['qas']:
                        faq_questions.append(qa['question'])
        except Exception:
            pass
for ms in _find('queries.dev.small.tsv') + _find('*.tsv'):
    try:
        q = pd.read_csv(ms, sep='\t', header=None, names=['id', 'query'])['query']
        faq_questions += q.dropna().astype(str).tolist()
    except Exception:
        pass
faq_questions = [q for q in faq_questions if isinstance(q, str) and len(q.strip()) > 8]
print(f'FAQ questions  : {len(faq_questions):,} (SQuAD + MS MARCO)')

## 7. Map Labels → the 5 Agent Domains

**Why this step?** Banking77's 77 fine-grained intents (and CFPB's issue types)
must be collapsed onto our five routing domains. We use a transparent,
**rule-based mapping** on the intent string — easy to audit and refine — then
attach CFPB narratives as `complaint` and the general questions as `faq`.

> This is a *starter* mapping. Refine the keyword rules to tune routing quality;
> the only hard requirement is that the final labels are exactly the 5 domains.

In [ ]:
# ============================================================
# SECTION 7: MAP EVERYTHING → {billing, technical, product, complaint, faq}
# ============================================================
def map_banking77(category: str) -> str:
    c = category.lower()
    if any(k in c for k in ['fee', 'charge', 'refund', 'payment', 'transfer',
                            'top_up', 'topping', 'withdrawal', 'balance',
                            'transaction', 'direct_debit', 'pending', 'receiving_money']):
        return 'billing'
    if any(k in c for k in ['pin', 'verify', 'identity', 'not_working', 'activate',
                            'linking', 'swallow', 'compromised', 'lost_or_stolen',
                            'passcode', 'contactless', 'terminate', 'edit_personal', 'atm']):
        return 'technical'
    if any(k in c for k in ['card_arrival', 'delivery', 'physical_card', 'virtual_card',
                            'spare_card', 'disposable', 'supported_cards', 'visa_or_mastercard',
                            'apple_pay', 'fiat_currency', 'exchange_rate', 'exchange_via',
                            'acceptance', 'about_to_expire', 'order_physical']):
        return 'product'
    return 'faq'

frames = []
if len(banking77):
    b = banking77.dropna(subset=['text', 'category']).copy()
    b['domain'] = b['category'].map(map_banking77)
    frames.append(b[['text', 'domain']])

if cfpb_texts:
    frames.append(pd.DataFrame({'text': cfpb_texts, 'domain': 'complaint'}))

if faq_questions:
    frames.append(pd.DataFrame({'text': faq_questions, 'domain': 'faq'}))

data = pd.concat(frames, ignore_index=True)
data['text'] = data['text'].astype(str).str.strip()
data = data[data['text'].str.len() > 0].drop_duplicates(subset='text')

# ── Balance the classes (cap per domain) so none dominates ──
CAP = int(data['domain'].value_counts().median())
CAP = max(1500, min(CAP, 6000))
balanced = (data.groupby('domain', group_keys=False)
                .apply(lambda g: g.sample(min(len(g), CAP), random_state=42)))
print('Class distribution after balancing (cap =', CAP, '):')
print(balanced['domain'].value_counts())
print(f'\n✅ Training set: {len(balanced):,} rows across {balanced["domain"].nunique()} domains')

## 8. Tokenize & Split

**Why this step?** DistilBERT needs integer token ids. We stratify a 90/10
train/validation split (so every domain is represented in both) and tokenize to
a fixed `max_length=128` — plenty for short support messages.

In [ ]:
# ============================================================
# SECTION 8: TOKENIZE + STRATIFIED SPLIT
# ============================================================
balanced['label'] = balanced['domain'].map(LABEL2ID)
train_df, val_df = train_test_split(
    balanced[['text', 'label']], test_size=0.10,
    stratify=balanced['label'], random_state=42,
)
print(f'train={len(train_df):,}  val={len(val_df):,}')

tokenizer = AutoTokenizer.from_pretrained(BASE_LM)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tok, batched=True)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False).map(tok, batched=True)
cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format('torch', columns=cols)
val_ds.set_format('torch', columns=cols)
print('✅ Tokenized.')

## 9. Fine-Tune DistilBERT (GPU)

**Why this step?** This produces Artifact 2. We load `distilbert-base-uncased`
with a fresh 5-way head and, crucially, set `id2label`/`label2id` to the domain
names so the saved model emits `billing`/`technical`/… directly — matching
`backend/agents/intent_classifier.py`. A small compatibility shim handles the
`eval_strategy` vs `evaluation_strategy` rename across transformers versions.

In [ ]:
# ============================================================
# SECTION 9: FINE-TUNE DistilBERT  ── (produces ARTIFACT 2)
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_LM, num_labels=len(DOMAINS), id2label=ID2LABEL, label2id=LABEL2ID,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_macro': f1_score(labels, preds, average='macro')}

# Compat shim: 'eval_strategy' (new) vs 'evaluation_strategy' (old)
import inspect
_params = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = 'eval_strategy' if 'eval_strategy' in _params else 'evaluation_strategy'
ta_kwargs = {
    'output_dir': '/kaggle/working/_train',
    'num_train_epochs': 3,
    'per_device_train_batch_size': 32,
    'per_device_eval_batch_size': 64,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'logging_steps': 50,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_macro',
    'fp16': torch.cuda.is_available(),
    'report_to': 'none',
    _eval_key: 'epoch',
}
training_args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
print('✅ Fine-tuning complete.')

## 10. Evaluate the Classifier

**Why this step?** Before shipping, confirm accuracy + macro-F1 and inspect the
per-domain report and confusion matrix. We also spot-check real queries and
verify the confidence range is sane for the production threshold
(`INTENT_CONFIDENCE_THRESHOLD = 0.4`).

In [ ]:
# ============================================================
# SECTION 10: EVALUATE
# ============================================================
metrics = trainer.evaluate()
print('Validation:', {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

pred = trainer.predict(val_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids
print('\n' + classification_report(y_true, y_pred, target_names=DOMAINS, digits=3))
print('Confusion matrix (rows=true, cols=pred):')
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=DOMAINS, columns=DOMAINS))

# Spot-check
clf = pipeline('text-classification', model=model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1, top_k=None)
for q in ['I was charged twice for my order',
          'I cannot log into my account',
          'What laptops do you sell?',
          'This is the worst service ever, I demand a manager',
          'What are your business hours?']:
    top = max(clf(q)[0], key=lambda x: x['score'])
    print(f'  {top["label"]:<10} {top["score"]:.2f}  <- {q}')

## 11. Save the Classifier  ──  **ARTIFACT 2**

**Why this step?** `save_model` + `save_pretrained` write the full Hugging Face
directory (`config.json`, weights, tokenizer files) that
`pipeline("text-classification", model=...)` loads in production.

In [ ]:
# ============================================================
# SECTION 11: SAVE CLASSIFIER  ── ARTIFACT 2
# ============================================================
trainer.save_model(CLF_OUT)       # config.json + model.safetensors
tokenizer.save_pretrained(CLF_OUT)  # tokenizer.json, vocab.txt, ...

print('✅ ARTIFACT 2 saved to', CLF_OUT)
for fn in sorted(os.listdir(CLF_OUT)):
    print('   -', fn, f'({os.path.getsize(os.path.join(CLF_OUT, fn))/1024:.1f} KB)')

## 12. Verify & Package Both Artifacts

**Why this step?** Prove — *inside the notebook* — that both artifacts reload and
work, exactly as production will use them. Then zip them for a one-click download
from the **Output** tab.

In [ ]:
# ============================================================
# SECTION 12a: VERIFY ARTIFACT 1 (reload FAISS + query)
# ============================================================
idx = faiss.read_index(os.path.join(FAISS_OUT, 'index.faiss'))
meta = pickle.load(open(os.path.join(FAISS_OUT, 'metadata.pkl'), 'rb'))
verifier = SentenceTransformer(EMBED_MODEL, device=DEVICE)

for q in ['what is the refund policy?', 'how do I install the product?']:
    qv = verifier.encode([q]).astype('float32')
    D, I = idx.search(qv, 3)
    print(f'\n🔎 {q}')
    for rank, i in enumerate(I[0]):
        print(f'   {rank+1}. [{meta[i]["source"]} p{meta[i]["page"]}] {meta[i]["text"][:70]}…')
print('\n✅ Artifact 1 verified.')

In [ ]:
# ============================================================
# SECTION 12b: VERIFY ARTIFACT 2 (reload from disk as production does)
# ============================================================
prod_clf = pipeline('text-classification', model=CLF_OUT, tokenizer=CLF_OUT,
                    device=0 if torch.cuda.is_available() else -1, top_k=None)
sample = 'I paid but my premium is still locked'
scores = {d['label']: round(d['score'], 3) for d in prod_clf(sample)[0]}
print('sample:', sample)
print('scores:', scores)
assert set(scores) == set(DOMAINS), 'labels must be the 5 domains!'
print('✅ Artifact 2 verified — labels are the 5 domains.')

In [ ]:
# ============================================================
# SECTION 12c: PACKAGE FOR DOWNLOAD
# ============================================================
import shutil
shutil.make_archive('/kaggle/working/faiss_index', 'zip', FAISS_OUT)
shutil.make_archive('/kaggle/working/intent_classifier', 'zip', CLF_OUT)
print('✅ Zipped:')
for z in ['/kaggle/working/faiss_index.zip', '/kaggle/working/intent_classifier.zip']:
    print(f'   {z}  ({os.path.getsize(z)/1024/1024:.2f} MB)')
print('\n🎉 Both artifacts built, verified, and packaged.')

## Next Steps — Deploy to Production

1. **Save Version → Save & Run All (Commit)** to execute top-to-bottom on Kaggle
   and snapshot `/kaggle/working/`.
2. From the completed version's **Output** tab, download:
   - `faiss_index.zip`  → unzip into `vectorstore/faiss_index/` (`index.faiss`, `metadata.pkl`)
   - `intent_classifier.zip` → unzip into `backend/models/intent_classifier/`
3. Keep `ROUTING_MODE=local` in `.env` (the default). If the artifact is absent
   or fails to load, the backend **auto-falls back to the LLM**, so deploys are
   safe even before the artifact exists.

> **Note:** both artifact folders are **git-ignored** (build outputs, not source
> — see `.gitignore` and `backend/models/intent_classifier/README.md`).
> Re-generate them from this notebook rather than committing them.

### ✅ Result
Routing now runs as a **local forward pass** (0 LLM calls, no 429s); the LLM is
reserved for the high-value final response. This is the Kaggle Factory payoff:
decoupled workloads, fast/cheap inference, and a lightweight, stateless server.

### Hurrah! The TechMart Kaggle Factory notebook is complete.
Two artifacts — **RAG index** and **intent router** — are ready to power the
production Multi-Agent Assistant.